# 02. Multi-Layer Perceptron (MLP) with MNIST

## 📌 실습 개요

이번 실습에서는 **MNIST 손글씨 숫자 데이터셋**을 이용하여  
TensorFlow/Keras 기반의 **다층 퍼셉트론(MLP)** 을 구현합니다.

**MNIST 데이터 → 데이터 확인 → 정규화 → Flatten → MLP 구성 → 학습 → 평가 → 예측 → 오답 분석**

## 1. 라이브러리 불러오기

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

print("TensorFlow Version:", tf.__version__)

## 2. MNIST Dataset

- Training Data: 60,000
- Test Data: 10,000
- Image Size: 28 × 28
- Class: 10개 (0~9)

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape :", x_test.shape)
print("y_test shape :", y_test.shape)

## 3. 데이터 확인

MNIST 한 장의 이미지는 `28 × 28` 크기의 숫자 배열입니다.  
각 픽셀 값은 `0 ~ 255` 범위를 가집니다.

In [ ]:
print("첫 번째 이미지 Label:", y_train[0])
print("Pixel Min:", x_train[0].min())
print("Pixel Max:", x_train[0].max())

plt.imshow(x_train[0], cmap="gray")
plt.title(f"Label: {y_train[0]}")
plt.axis("off")
plt.show()

### 여러 이미지 확인

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 3.1 숫자별 데이터 분포 확인

MNIST 학습 데이터에 숫자 `0 ~ 9`가 각각 몇 개씩 포함되어 있는지 확인합니다.

`np.unique(..., return_counts=True)`를 사용하면 각 Label별 데이터 개수를 구할 수 있습니다.

막대그래프를 통해 **클래스별 데이터 분포가 비교적 균형적인지** 확인할 수 있습니다.


In [ ]:
labels, counts = np.unique(y_train, return_counts=True)

print("숫자별 학습 데이터 개수")
for label, count in zip(labels, counts):
    print(f"{label}: {count}")

plt.figure(figsize=(8, 5))
plt.bar(labels, counts)

plt.xticks(labels)
plt.xlabel("Digit")
plt.ylabel("Count")
plt.title("MNIST Training Data Distribution")
plt.grid(axis="y", alpha=0.3)

plt.show()


## 4. 데이터 정규화

픽셀 값을 `0 ~ 255`에서 `0 ~ 1` 범위로 변환합니다.

```text
정규화된 값 = x / 255
```

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Min:", x_train.min())
print("Max:", x_train.max())

## 5. Flatten

Dense Layer는 1차원 입력을 사용하므로 `28 × 28` 이미지를 784개의 값으로 펼칩니다.

```text
28 × 28 → 784
```

## 6. MLP 모델 구성

```text
28 × 28
   ↓
Flatten
   ↓
784
   ↓
Dense(128, ReLU)
   ↓
Dense(64, ReLU)
   ↓
Dense(10, Softmax)
```

In [ ]:
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

model.summary()

## 7. 모델 컴파일

- Optimizer: Adam
- Loss: Sparse Categorical Crossentropy
- Metric: Accuracy

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 8. 모델 학습

- `epochs=10`
- `batch_size=128`
- `validation_split=0.2`

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2
)

## 9. Accuracy 그래프

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training / Validation Accuracy")
plt.legend()
plt.grid()
plt.show()

## 10. Loss 그래프

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid()
plt.show()

## 11. 테스트 데이터 평가

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

## 12. 실제 숫자 예측

Softmax 출력 중 가장 큰 값의 위치를 `argmax()`로 선택합니다.

In [ ]:
predictions = model.predict(x_test[:10], verbose=0)

print("첫 번째 이미지의 클래스별 확률:")
print(predictions[0])

print("예측:", np.argmax(predictions[0]))
print("정답:", y_test[0])

## 13. 예측 결과 시각화

In [ ]:
plt.figure(figsize=(12, 6))

predictions = model.predict(x_test[:15], verbose=0)

for i in range(15):
    pred = np.argmax(predictions[i])
    actual = y_test[i]

    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    plt.title(f"Pred: {pred}\nTrue: {actual}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 14. 오답 데이터 확인

In [ ]:
predictions = model.predict(x_test, verbose=0)
pred_labels = np.argmax(predictions, axis=1)

wrong_indices = np.where(pred_labels != y_test)[0]

print("전체 테스트 데이터:", len(y_test))
print("오답 개수:", len(wrong_indices))

In [ ]:
plt.figure(figsize=(12, 6))

for i, idx in enumerate(wrong_indices[:15]):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[idx], cmap="gray")
    plt.title(f"Pred: {pred_labels[idx]}\nTrue: {y_test[idx]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# 실습 문제

### 실습 1
첫 번째 은닉층의 뉴런 수를 `32`, `64`, `128`, `256`으로 변경하고 정확도를 비교하세요.

### 실습 2
다음 구조로 변경하세요.

```text
Flatten
Dense(256, ReLU)
Dense(128, ReLU)
Dense(64, ReLU)
Dense(10, Softmax)
```

### 실습 3
`epochs=5`, `10`, `20`으로 변경하여 결과를 비교하세요.

### 실습 4
ReLU를 Sigmoid로 변경하고 학습 결과를 비교하세요. 